# Footprint Heatmap — Mapa nacional de usuarios, GMV y cobertura

Notebook chico y autocontenido (no depende de `footprint_ml_store_explorer.ipynb`).
Un solo mapa con:

- Heatmap de densidad de usuarios (todo el pais).
- Heatmap de GMV PeYa y heatmap de GMV Dmart, en capas separadas.
- Poligono real de cobertura de cada dark store (`qc_coverage_partner_shapes.partner_shape`);
  circulo de `SERVICE_RADIUS_KM` como fallback si esa tienda no tiene shape cargado ese mes.
- Marcador por ciudad y por tienda, cada uno con un popup de metricas pre-calculadas
  (sin backend vivo — todo se arma una vez al correr el notebook).

**Principio de diseño:** con ~7M de usuarios, nunca se baja usuario por usuario. La grilla de
heatmap se agrega **dentro de BigQuery** (Bloque 1a) antes de traer nada a Python.

| Bloque | Nombre | Toca BQ |
|---|---|---|
| 0 | Configuracion global | No |
| 1 | Ingesta (grilla + ciudades + tiendas) | **Si — una sola vez** |
| 2 | Mapa | No |


---
## Bloque 0 — Configuracion global

Pais fijo (sin selector), mes puntual (snapshot), tamano de celda de la grilla y radio de
cobertura fallback. Cambiar mercado o mes = cambiar esta celda.


In [1]:
# =============================================================================
# BLOQUE 0 — CONFIGURACION GLOBAL
# =============================================================================
!pip install -q google-cloud-bigquery folium

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception as e:
    print(f'[warn] No estamos en Colab: {e}')

# =============================================================================
# PARAMETROS
# =============================================================================
PROJECT_ID    = 'peya-food-and-groceries'   # proyecto de facturacion de BQ
COUNTRY_ID    = 3              # 3 = Argentina — pais fijo, sin selector
COUNTRY_LABEL = 'Argentina'    # solo para titulos/prints
TARGET_MONTH  = '2026-05-01'   # mes puntual (snapshot), no acumulado

GRID_CELL_DEG      = 0.005   # lado de celda de la grilla de heatmap (~500 m)
SERVICE_RADIUS_KM  = 2.5     # circulo de cobertura fallback si la tienda no tiene poligono ese mes
MAX_CELDAS_HEATMAP = 20000   # guardarrail: si la grilla agregada supera esto, se recorta por GMV

print('Bloque 0 OK — configuracion cargada.')
print(f'  Pais / mes   : {COUNTRY_LABEL} ({COUNTRY_ID}) / {TARGET_MONTH}')
print(f'  Celda grilla : {GRID_CELL_DEG}° (~{GRID_CELL_DEG * 111_000:.0f} m de lado)')
print(f'  Radio fallback de cobertura: {SERVICE_RADIUS_KM} km')


Bloque 0 OK — configuracion cargada.
  Pais / mes   : Argentina (3) / 2026-05-01
  Celda grilla : 0.005° (~555 m de lado)
  Radio fallback de cobertura: 2.5 km


---
## Bloque 1 — Ingesta (4 queries, corre una sola vez)

**Guardarrail de costo:** el flag `_BQ_YA_CORRIO` evita re-ejecuciones accidentales, igual que
en el Footprint ML Explorer.

- **1a — Grilla agregada:** agrupa los ~7M usuarios por celda de `GRID_CELL_DEG` grados
  (redondeo a multiplo de celda, no solo decimales) **dentro de BigQuery**. De millones de
  filas de usuario se pasa a, como mucho, unos pocos miles de celdas — eso es lo unico que
  baja a Python para los 3 heatmaps (usuarios, GMV PeYa, GMV Dmart).
- **1b — Metricas por ciudad:** un row por ciudad con AOV, % Plus, % Core, % High y
  penetracion Dmart, mas su centroide (para el marcador).
- **1c — Tiendas + poligono de cobertura:** un row por dark store con penetracion, MAUs
  cubiertos, ordenes, AFV y basket size, mas su punto (`store_lat`/`store_lon`, siempre
  presente) y el poligono real de `qc_coverage_partner_shapes.partner_shape` como GeoJSON
  (`shape_geojson`, `null` si esa tienda no tiene shape cargado ese mes — el Bloque 2 cae a un
  circulo en ese caso).
- **1d — Metricas por zona:** mismo patron que 1b pero a nivel `zone_name` (mas fino que
  ciudad). No hay poligono real de zona en BQ (solo puntos de usuario etiquetados), asi que se
  muestra como marcador en el centroide, igual que las ciudades — no como area.

> Nota: 1c y 1d son queries nuevas (adaptadas de `QUERY_WAREHOUSE`/`QUERY_CITY` del otro
> notebook) — conviene revisar el primer resultado en BQ antes de confiar en el mapa completo.


In [2]:
# =============================================================================
# BLOQUE 1 — INGESTA (4 queries agregadas en BQ; corre una sola vez por sesion)
# =============================================================================
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

if '_BQ_YA_CORRIO' in globals() and _BQ_YA_CORRIO:
    print('BQ ya corrio en esta sesion. Reusando df_grid / df_ciudades / df_tiendas / df_zonas en memoria.')
else:
    # -------------------------------------------------------------------------
    # 1a) Grilla agregada para los heatmaps (usuarios, GMV PeYa, GMV Dmart)
    # -------------------------------------------------------------------------
    print('Ejecutando query 1/4 — grilla agregada ...')
    QUERY_GRID = f"""
    WITH coverage AS (
      SELECT
        cl.customer_id,
        ROUND(ST_Y(cl.customer_geogpoint) / {GRID_CELL_DEG}) * {GRID_CELL_DEG} AS lat_bin,
        ROUND(ST_X(cl.customer_geogpoint) / {GRID_CELL_DEG}) * {GRID_CELL_DEG} AS lon_bin
      FROM `peya-food-and-groceries.automated_tables_reports.qc_coverage_customer_location` cl
      WHERE cl.month_date = '{TARGET_MONTH}'
        AND cl.country_id = {COUNTRY_ID}
        AND cl.customer_geogpoint IS NOT NULL
    ),

    peya_gmv AS (
      SELECT
        o.user.customer_id,
        SUM(o.gmv.gmv_user / monthly_rate_eu) AS gmv_peya
      FROM `peya-datamarts-pro.dm_order_profitability.fact_order_profitability` o
      WHERE DATE_TRUNC(o.registered_date, MONTH) = '{TARGET_MONTH}'
        AND o.order_status = 'CONFIRMED'
        AND o.country_id = {COUNTRY_ID}
      GROUP BY 1
    ),

    dmart_gmv AS (
      SELECT
        CAST(customer_id AS STRING) AS customer_id,
        SUM(total_amount_net_with_discount_eur) AS gmv_dmart
      FROM `peya-datamarts-pro.dm_dmarts.customer_orders_sku_level`
      WHERE DATE_TRUNC(order_date, MONTH) = '{TARGET_MONTH}'
        AND state = 'CONFIRMED'
        AND CAST(country_id AS INT64) = {COUNTRY_ID}
      GROUP BY 1
    )

    SELECT
      c.lat_bin,
      c.lon_bin,
      COUNT(DISTINCT c.customer_id) AS usuarios,
      SUM(COALESCE(p.gmv_peya, 0))  AS gmv_peya,
      SUM(COALESCE(d.gmv_dmart, 0)) AS gmv_dmart
    FROM coverage c
    LEFT JOIN peya_gmv  p ON c.customer_id = p.customer_id
    LEFT JOIN dmart_gmv d ON c.customer_id = d.customer_id
    GROUP BY 1, 2
    """
    df_grid = client.query(QUERY_GRID).to_dataframe()
    print(f'OK. {len(df_grid):,} celdas de grilla.')

    # -------------------------------------------------------------------------
    # 1b) Metricas por ciudad (para el popup de ciudad)
    # -------------------------------------------------------------------------
    print('Ejecutando query 2/4 — metricas por ciudad ...')
    QUERY_CIUDADES = f"""
    WITH customer_with_coverage AS (
      SELECT DISTINCT
        cl.customer_id,
        cl.city_name_lz,
        ST_Y(cl.customer_geogpoint) AS lat,
        ST_X(cl.customer_geogpoint) AS lon
      FROM `peya-food-and-groceries.automated_tables_reports.qc_coverage_customer_location` cl
      WHERE cl.month_date = '{TARGET_MONTH}'
        AND cl.country_id = {COUNTRY_ID}
        AND cl.customer_geogpoint IS NOT NULL
    ),

    user_profile AS (
      SELECT
        du.customer_id,
        lc.country_id,
        MAX(CASE WHEN lc.user_income = 'High' THEN 1 ELSE 0 END) AS is_high_income_user,
        MAX(CASE WHEN lc.subsegment = 'Core'  THEN 1 ELSE 0 END) AS is_core_user
      FROM `peya-bi-tools-pro.il_qcommerce.user_lifecycle` lc
      INNER JOIN `peya-bi-tools-pro.il_core.dim_user` du
        ON lc.user_id = du.user_id AND lc.country_id = du.country_id
      GROUP BY 1, 2
    ),

    dmart_metrics AS (
      SELECT
        CAST(customer_id AS STRING) AS customer_id,
        COUNT(DISTINCT order_id) AS dmart_orders
      FROM `peya-datamarts-pro.dm_dmarts.customer_orders_sku_level`
      WHERE DATE_TRUNC(order_date, MONTH) = '{TARGET_MONTH}'
        AND state = 'CONFIRMED'
        AND CAST(country_id AS INT64) = {COUNTRY_ID}
      GROUP BY 1
    ),

    fact_orders_aggregated AS (
      SELECT
        CAST(o.user.customer_id AS STRING) AS customer_id,
        SUM(o.gmv.gmv_user / monthly_rate_eu) AS customer_value,
        COUNT(DISTINCT o.order_id) AS orders,
        MAX(CASE WHEN o.user.is_user_plus THEN 1 ELSE 0 END) AS is_user_plus_flag
      FROM `peya-datamarts-pro.dm_order_profitability.fact_order_profitability` o
      WHERE DATE_TRUNC(o.registered_date, MONTH) = '{TARGET_MONTH}'
        AND o.order_status = 'CONFIRMED'
        AND o.country_id = {COUNTRY_ID}
      GROUP BY 1
    ),

    maus_perfo AS (
      SELECT
        cl.customer_id,
        cl.city_name_lz,
        cl.lat,
        cl.lon,
        MAX(foa.is_user_plus_flag) AS is_user_plus,
        MAX(up.is_high_income_user) AS high_user,
        MAX(up.is_core_user) AS core_user,
        SUM(foa.customer_value) AS customer_value,
        SUM(foa.orders) AS orders,
        MAX(CASE WHEN dm.dmart_orders IS NOT NULL THEN 1 ELSE 0 END) AS is_dmart_user
      FROM customer_with_coverage cl
      LEFT JOIN fact_orders_aggregated foa ON cl.customer_id = foa.customer_id
      LEFT JOIN user_profile up ON cl.customer_id = up.customer_id
      LEFT JOIN dmart_metrics dm ON cl.customer_id = dm.customer_id
      GROUP BY 1, 2, 3, 4
    )

    SELECT
      city_name_lz,
      AVG(lat) AS lat_centro,
      AVG(lon) AS lon_centro,
      COUNT(DISTINCT customer_id) AS peya_users,
      ROUND(SAFE_DIVIDE(SUM(customer_value), COUNT(DISTINCT customer_id)), 2) AS avg_customer_value,
      ROUND(SAFE_DIVIDE(SUM(orders), COUNT(DISTINCT customer_id)), 2) AS freq,
      ROUND(SAFE_DIVIDE(SUM(is_user_plus), COUNT(DISTINCT customer_id)), 4) AS plus_users_pct,
      ROUND(SAFE_DIVIDE(SUM(core_user), COUNT(DISTINCT customer_id)), 4) AS core_users_pct,
      ROUND(SAFE_DIVIDE(SUM(high_user), COUNT(DISTINCT customer_id)), 4) AS high_users_pct,
      ROUND(SAFE_DIVIDE(SUM(is_dmart_user), COUNT(DISTINCT customer_id)), 4) AS groceries_penetration
    FROM maus_perfo
    WHERE city_name_lz IS NOT NULL
    GROUP BY city_name_lz
    ORDER BY peya_users DESC
    """
    df_ciudades = client.query(QUERY_CIUDADES).to_dataframe()
    print(f'OK. {len(df_ciudades):,} ciudades.')

    # -------------------------------------------------------------------------
    # 1c) Tiendas + poligono real de cobertura (para el popup/mapa)
    # -------------------------------------------------------------------------
    print('Ejecutando query 3/4 — tiendas y poligonos de cobertura ...')
    QUERY_TIENDAS = f"""
    WITH warehouse_top_partner AS (
      SELECT warehouse_id, vendor_id
      FROM (
        SELECT
          warehouse_id,
          vendor_id,
          ROW_NUMBER() OVER (
            PARTITION BY warehouse_id
            ORDER BY COUNT(DISTINCT order_id) DESC
          ) AS rn
        FROM `peya-datamarts-pro.dm_dmarts.customer_orders_sku_level`
        WHERE DATE_TRUNC(order_date, MONTH) = '{TARGET_MONTH}'
          AND state = 'CONFIRMED'
          AND CAST(country_id AS INT64) = {COUNTRY_ID}
        GROUP BY warehouse_id, vendor_id
      )
      WHERE rn = 1
    ),

    dmart_partners AS (
      SELECT
        dp.warehouse_id,
        CAST(dp.partner_id AS STRING) AS partner_id,
        dp.warehouse_name,
        ST_GEOGPOINT(dp.address.longitude, dp.address.latitude) AS store_geopoint,
        ST_Y(ST_GEOGPOINT(dp.address.longitude, dp.address.latitude)) AS store_lat,
        ST_X(ST_GEOGPOINT(dp.address.longitude, dp.address.latitude)) AS store_lon
      FROM `peya-bi-tools-pro.il_core.dim_partner` dp
      INNER JOIN warehouse_top_partner wtp
        ON CAST(dp.partner_id AS STRING) = CAST(wtp.vendor_id AS STRING)
      WHERE dp.is_darkstore
    ),

    dmart_shapes AS (
      SELECT
        ps.partner_id,
        ps.partner_shape,
        dp.warehouse_id,
        dp.store_geopoint
      FROM `peya-food-and-groceries.automated_tables_reports.qc_coverage_partner_shapes` ps
      INNER JOIN dmart_partners dp ON CAST(ps.partner_id AS STRING) = dp.partner_id
      WHERE ps.month_date = '{TARGET_MONTH}'
    ),

    warehouse_shape AS (
      SELECT
        warehouse_id,
        ANY_VALUE(ST_ASGEOJSON(partner_shape)) AS shape_geojson
      FROM dmart_shapes
      GROUP BY warehouse_id
    ),

    dmart_cover_all AS (
      SELECT
        CAST(cl.customer_id AS STRING) AS customer_id,
        cs.warehouse_id,
        cs.store_geopoint,
        cl.customer_geogpoint,
        COUNT(*) OVER (PARTITION BY CAST(cl.customer_id AS STRING)) AS num_covering_stores
      FROM `peya-food-and-groceries.automated_tables_reports.qc_coverage_customer_location` cl
      JOIN dmart_shapes cs
        ON ST_WITHIN(cl.customer_geogpoint, cs.partner_shape)
      WHERE cl.month_date = '{TARGET_MONTH}'
        AND cl.country_id = {COUNTRY_ID}
    ),

    dmart_cover_maus AS (
      SELECT customer_id, warehouse_id
      FROM (
        SELECT *,
          CASE
            WHEN num_covering_stores = 1 THEN 1
            ELSE ROW_NUMBER() OVER (
              PARTITION BY customer_id
              ORDER BY ST_DISTANCE(customer_geogpoint, store_geopoint)
            )
          END AS store_rank
        FROM dmart_cover_all
      )
      WHERE store_rank = 1
    ),

    dmart_metrics AS (
      SELECT
        CAST(customer_id AS STRING) AS customer_id,
        COUNT(DISTINCT order_id) AS dmart_orders,
        SUM(total_amount_net_with_discount_eur) AS dmart_net_sales,
        SUM(fulfilled_quantity) AS dmart_units
      FROM `peya-datamarts-pro.dm_dmarts.customer_orders_sku_level`
      WHERE DATE_TRUNC(order_date, MONTH) = '{TARGET_MONTH}'
        AND state = 'CONFIRMED'
        AND CAST(country_id AS INT64) = {COUNTRY_ID}
      GROUP BY 1
    ),

    perfo AS (
      SELECT
        cm.warehouse_id,
        cm.customer_id,
        dm.dmart_orders,
        dm.dmart_net_sales,
        dm.dmart_units
      FROM dmart_cover_maus cm
      LEFT JOIN dmart_metrics dm ON cm.customer_id = dm.customer_id
    ),

    aggregated AS (
      SELECT
        warehouse_id,
        COUNT(DISTINCT customer_id) AS users,
        COUNT(DISTINCT CASE WHEN dmart_orders IS NOT NULL THEN customer_id END) AS dmart_users,
        SUM(COALESCE(dmart_orders, 0)) AS dmart_orders,
        SUM(COALESCE(dmart_net_sales, 0)) AS dmart_net_sales,
        SUM(COALESCE(dmart_units, 0)) AS dmart_units
      FROM perfo
      GROUP BY warehouse_id
    )

    SELECT
      dp.warehouse_id,
      dp.warehouse_name,
      dp.store_lat,
      dp.store_lon,
      ws.shape_geojson,
      a.users                                                  AS users_cubiertos,
      a.dmart_orders                                           AS ordenes,
      ROUND(SAFE_DIVIDE(a.dmart_users, a.users), 4)            AS penetracion,
      ROUND(SAFE_DIVIDE(a.dmart_net_sales, a.dmart_orders), 2) AS afv,
      ROUND(SAFE_DIVIDE(a.dmart_units, a.dmart_orders), 2)     AS basket_size
    FROM dmart_partners dp
    LEFT JOIN warehouse_shape ws ON dp.warehouse_id = ws.warehouse_id
    LEFT JOIN aggregated a ON dp.warehouse_id = a.warehouse_id
    WHERE a.users IS NOT NULL
    """
    df_tiendas = client.query(QUERY_TIENDAS).to_dataframe()
    print(f'OK. {len(df_tiendas):,} tiendas '
          f'({df_tiendas["shape_geojson"].notna().sum():,} con poligono real).')

    # -------------------------------------------------------------------------
    # 1d) Metricas por zona (para el popup de zona; sin poligono, solo centroide)
    # -------------------------------------------------------------------------
    print('Ejecutando query 4/4 — metricas por zona ...')
    QUERY_ZONAS = f"""
    WITH customer_with_coverage AS (
      SELECT DISTINCT
        cl.customer_id,
        cl.city_name_lz,
        cl.zone_name,
        ST_Y(cl.customer_geogpoint) AS lat,
        ST_X(cl.customer_geogpoint) AS lon
      FROM `peya-food-and-groceries.automated_tables_reports.qc_coverage_customer_location` cl
      WHERE cl.month_date = '{TARGET_MONTH}'
        AND cl.country_id = {COUNTRY_ID}
        AND cl.customer_geogpoint IS NOT NULL
    ),

    user_profile AS (
      SELECT
        du.customer_id,
        lc.country_id,
        MAX(CASE WHEN lc.user_income = 'High' THEN 1 ELSE 0 END) AS is_high_income_user,
        MAX(CASE WHEN lc.subsegment = 'Core'  THEN 1 ELSE 0 END) AS is_core_user
      FROM `peya-bi-tools-pro.il_qcommerce.user_lifecycle` lc
      INNER JOIN `peya-bi-tools-pro.il_core.dim_user` du
        ON lc.user_id = du.user_id AND lc.country_id = du.country_id
      GROUP BY 1, 2
    ),

    dmart_metrics AS (
      SELECT
        CAST(customer_id AS STRING) AS customer_id,
        COUNT(DISTINCT order_id) AS dmart_orders
      FROM `peya-datamarts-pro.dm_dmarts.customer_orders_sku_level`
      WHERE DATE_TRUNC(order_date, MONTH) = '{TARGET_MONTH}'
        AND state = 'CONFIRMED'
        AND CAST(country_id AS INT64) = {COUNTRY_ID}
      GROUP BY 1
    ),

    fact_orders_aggregated AS (
      SELECT
        CAST(o.user.customer_id AS STRING) AS customer_id,
        SUM(o.gmv.gmv_user / monthly_rate_eu) AS customer_value,
        COUNT(DISTINCT o.order_id) AS orders,
        MAX(CASE WHEN o.user.is_user_plus THEN 1 ELSE 0 END) AS is_user_plus_flag
      FROM `peya-datamarts-pro.dm_order_profitability.fact_order_profitability` o
      WHERE DATE_TRUNC(o.registered_date, MONTH) = '{TARGET_MONTH}'
        AND o.order_status = 'CONFIRMED'
        AND o.country_id = {COUNTRY_ID}
      GROUP BY 1
    ),

    maus_perfo AS (
      SELECT
        cl.customer_id,
        cl.city_name_lz,
        cl.zone_name,
        cl.lat,
        cl.lon,
        MAX(foa.is_user_plus_flag) AS is_user_plus,
        MAX(up.is_high_income_user) AS high_user,
        MAX(up.is_core_user) AS core_user,
        SUM(foa.customer_value) AS customer_value,
        SUM(foa.orders) AS orders,
        MAX(CASE WHEN dm.dmart_orders IS NOT NULL THEN 1 ELSE 0 END) AS is_dmart_user
      FROM customer_with_coverage cl
      LEFT JOIN fact_orders_aggregated foa ON cl.customer_id = foa.customer_id
      LEFT JOIN user_profile up ON cl.customer_id = up.customer_id
      LEFT JOIN dmart_metrics dm ON cl.customer_id = dm.customer_id
      GROUP BY 1, 2, 3, 4, 5
    )

    SELECT
      zone_name,
      city_name_lz,
      AVG(lat) AS lat_centro,
      AVG(lon) AS lon_centro,
      COUNT(DISTINCT customer_id) AS peya_users,
      ROUND(SAFE_DIVIDE(SUM(customer_value), COUNT(DISTINCT customer_id)), 2) AS avg_customer_value,
      ROUND(SAFE_DIVIDE(SUM(orders), COUNT(DISTINCT customer_id)), 2) AS freq,
      ROUND(SAFE_DIVIDE(SUM(is_user_plus), COUNT(DISTINCT customer_id)), 4) AS plus_users_pct,
      ROUND(SAFE_DIVIDE(SUM(core_user), COUNT(DISTINCT customer_id)), 4) AS core_users_pct,
      ROUND(SAFE_DIVIDE(SUM(high_user), COUNT(DISTINCT customer_id)), 4) AS high_users_pct,
      ROUND(SAFE_DIVIDE(SUM(is_dmart_user), COUNT(DISTINCT customer_id)), 4) AS groceries_penetration
    FROM maus_perfo
    WHERE zone_name IS NOT NULL
    GROUP BY zone_name, city_name_lz
    ORDER BY peya_users DESC
    """
    df_zonas = client.query(QUERY_ZONAS).to_dataframe()
    print(f'OK. {len(df_zonas):,} zonas.')

    _BQ_YA_CORRIO = True
    print('\nIngesta completa. BigQuery no se vuelve a tocar en toda la sesion.')

print(f'\ndf_grid     : {len(df_grid):,} celdas')
print(f'df_ciudades : {len(df_ciudades):,} ciudades')
print(f'df_tiendas  : {len(df_tiendas):,} tiendas')
print(f'df_zonas    : {len(df_zonas):,} zonas')
display(df_grid.head(3))
display(df_ciudades.head(3))
display(df_tiendas.head(3))
display(df_zonas.head(3))


Ejecutando query 1/4 — grilla agregada ...
OK. 21,488 celdas de grilla.
Ejecutando query 2/4 — metricas por ciudad ...
OK. 57 ciudades.
Ejecutando query 3/4 — tiendas y poligonos de cobertura ...
OK. 69 tiendas (69 con poligono real).
Ejecutando query 4/4 — metricas por zona ...
OK. 131 zonas.

Ingesta completa. BigQuery no se vuelve a tocar en toda la sesion.

df_grid     : 21,488 celdas
df_ciudades : 57 ciudades
df_tiendas  : 69 tiendas
df_zonas    : 131 zonas


,lat_bin,lon_bin,usuarios,gmv_peya,gmv_dmart
0,-34.570,-58.585,31,1388.876879,103.80
1,-31.435,-64.145,70,1606.502740,69.41
2,-34.705,-58.600,56,1251.023184,114.18


,city_name_lz,lat_centro,lon_centro,peya_users,avg_customer_value,freq,plus_users_pct,core_users_pct,high_users_pct,groceries_penetration
0,Buenos aires,-34.583866,-58.460835,842402,74.81,4.59,0.2988,0.6089,0.8756,0.2841
1,Buenos aires norte,-34.460245,-58.724052,316118,71.79,4.48,0.2955,0.4616,0.7089,0.3294
2,Gba sur,-34.752028,-58.352237,315552,66.60,4.39,0.2723,0.5195,0.7809,0.3124


,warehouse_id,warehouse_name,store_lat,store_lon,shape_geojson,users_cubiertos,ordenes,penetracion,afv,basket_size
0,92568ae9-592d-4a52-926a-982c3c796683,AR_80_Pacheco,-34.459805,-58.635020,"{ ""type"": ""Polygon"", ""coordinates"": [ [ [-58.6...",38520,39285,0.3581,15.27,9.020000000
1,b5462f70-c932-4a0e-9436-ecf7afa67013,AR_277_Berazategui,-34.773566,-58.209174,"{ ""type"": ""Polygon"", ""coordinates"": [ [ [-58.2...",16274,20924,0.4061,12.95,8.220000000
2,fb5322ef-c028-4b74-9299-c8101b444c42,AR_179_Tucuman Centro II,-26.814468,-65.223932,"{ ""type"": ""Polygon"", ""coordinates"": [ [ [-65.2...",33503,29760,0.3371,13.10,8.290000000


,zone_name,city_name_lz,lat_centro,lon_centro,peya_users,avg_customer_value,freq,plus_users_pct,core_users_pct,high_users_pct,groceries_penetration
0,Palermo,Buenos aires,-34.593720,-58.408284,200065,68.70,4.39,0.2754,0.5549,0.8772,0.2500
1,Rosario,Rosario,-32.946506,-60.656428,143376,57.33,4.19,0.2549,0.5520,0.8576,0.1496
2,Villa del parque,Buenos aires,-34.598930,-58.488524,126604,81.29,4.90,0.3133,0.6674,0.8990,0.2948


---
## Bloque 2 — Mapa

Un solo `folium.Map` con 6 capas togglables (`LayerControl`):

1. Densidad de usuarios (heatmap).
2. GMV PeYa (heatmap).
3. GMV Dmart (heatmap).
4. Ciudades — un marcador por ciudad en su centroide, con popup de AOV/% Plus/% Core/%
   High/penetracion Dmart.
5. Zonas — un marcador por zona (`zone_name`) en su centroide, mismo popup que ciudades pero
   a grano mas fino. Sin poligono real (no existe en BQ para zonas, a diferencia de tiendas).
6. Tiendas — poligono real de cobertura (o circulo de `SERVICE_RADIUS_KM` si esa tienda no
   tiene shape ese mes) + marcador con popup de penetracion, MAUs cubiertos, ordenes, AFV y
   basket size.

**Contraste del heatmap:** los pesos (`usuarios`, `gmv_peya`, `gmv_dmart`) se recortan al
percentil 97 antes de graficar — sin este recorte, unas pocas celdas con GMV/usuarios muy por
encima del resto "aplastan" la escala de color y todo el mapa se ve de un solo tono. Ademas se
usa un gradiente propio con mas escalones de color (azul -> celeste -> amarillo -> naranja ->
rojo) en vez del default de Leaflet.heat.

Todos los popups estan pre-calculados al armar el mapa (no hay backend vivo ni click-anywhere
generico) — son suficientes para "click en una tienda", "click en una ciudad" o "click en una
zona", que es lo que pediste.


In [3]:
# =============================================================================
# BLOQUE 2 — MAPA (heatmaps de usuarios/GMV + tiendas + ciudades + zonas)
# =============================================================================

# --- Guardarrail: recortar la grilla si es demasiado grande para el navegador ---
df_grid_mapa = df_grid.copy()
if len(df_grid_mapa) > MAX_CELDAS_HEATMAP:
    df_grid_mapa = df_grid_mapa.nlargest(MAX_CELDAS_HEATMAP, 'gmv_peya')
    print(f'[nota] Grilla recortada a las {MAX_CELDAS_HEATMAP:,} celdas de mayor GMV PeYa '
          f'(de {len(df_grid):,} originales).')

# --- Contraste del heatmap: recorte de percentil + gradiente con mas escalones ------------
# Sin el recorte, unas pocas celdas extremas hacen que Leaflet.heat normalice todo el resto
# a un mismo tono (efecto "mapa de un solo color" que se ve en la captura).
HEAT_PERCENTILE_CLIP = 97

def _clip_pct(serie, pct=HEAT_PERCENTILE_CLIP):
    limite = np.percentile(serie, pct)
    return serie.clip(upper=limite)

df_grid_mapa['usuarios_clip']  = _clip_pct(df_grid_mapa['usuarios'])
df_grid_mapa['gmv_peya_clip']  = _clip_pct(df_grid_mapa['gmv_peya'])
df_grid_mapa['gmv_dmart_clip'] = _clip_pct(df_grid_mapa['gmv_dmart'])

HEAT_GRADIENT = {
    0.2:  '#313695',  # azul oscuro (frio)
    0.4:  '#4575b4',
    0.6:  '#74add1',
    0.75: '#fee090',  # amarillo
    0.9:  '#f46d43',  # naranja
    1.0:  '#a50026',  # rojo (caliente)
}

lat_c = df_grid_mapa['lat_bin'].mean()
lon_c = df_grid_mapa['lon_bin'].mean()

mapa = folium.Map(location=[lat_c, lon_c], zoom_start=6, tiles='cartodbpositron')

# --- Capa 1: heatmap de densidad de usuarios ----------------------------------
fg_usuarios = folium.FeatureGroup(name='Densidad de usuarios', show=True)
HeatMap(df_grid_mapa[['lat_bin', 'lon_bin', 'usuarios_clip']].to_numpy().tolist(),
        radius=12, blur=18, min_opacity=0.25, gradient=HEAT_GRADIENT).add_to(fg_usuarios)
fg_usuarios.add_to(mapa)

# --- Capa 2: heatmap de GMV PeYa -----------------------------------------------
fg_gmv_peya = folium.FeatureGroup(name='GMV PeYa', show=False)
HeatMap(df_grid_mapa[['lat_bin', 'lon_bin', 'gmv_peya_clip']].to_numpy().tolist(),
        radius=12, blur=18, min_opacity=0.25, gradient=HEAT_GRADIENT).add_to(fg_gmv_peya)
fg_gmv_peya.add_to(mapa)

# --- Capa 3: heatmap de GMV Dmart -----------------------------------------------
fg_gmv_dmart = folium.FeatureGroup(name='GMV Dmart', show=False)
HeatMap(df_grid_mapa[['lat_bin', 'lon_bin', 'gmv_dmart_clip']].to_numpy().tolist(),
        radius=12, blur=18, min_opacity=0.25, gradient=HEAT_GRADIENT).add_to(fg_gmv_dmart)
fg_gmv_dmart.add_to(mapa)

# --- Capa 4: ciudades (marcador + popup con metricas) --------------------------
fg_ciudades = folium.FeatureGroup(name='Ciudades', show=False)
for r in df_ciudades.itertuples():
    if pd.isna(r.lat_centro) or pd.isna(r.lon_centro):
        continue
    popup_html = f"""
    <b>{r.city_name_lz}</b><br>
    Usuarios PeYa: {r.peya_users:,.0f}<br>
    AOV: EUR {r.avg_customer_value:,.2f}<br>
    Frecuencia: {r.freq:.2f}<br>
    Plus Users: {r.plus_users_pct:.1%}<br>
    Core Users: {r.core_users_pct:.1%}<br>
    High Users: {r.high_users_pct:.1%}<br>
    Penetracion Dmart: {r.groceries_penetration:.1%}
    """
    folium.Marker(
        [r.lat_centro, r.lon_centro],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color='blue', icon='info-sign'),
    ).add_to(fg_ciudades)
fg_ciudades.add_to(mapa)

# --- Capa 5: zonas (marcador + popup con metricas, mismo patron que ciudades) --
fg_zonas = folium.FeatureGroup(name='Zonas', show=False)
for r in df_zonas.itertuples():
    if pd.isna(r.lat_centro) or pd.isna(r.lon_centro):
        continue
    popup_html = f"""
    <b>{r.zone_name}</b> ({r.city_name_lz})<br>
    Usuarios PeYa: {r.peya_users:,.0f}<br>
    AOV: EUR {r.avg_customer_value:,.2f}<br>
    Frecuencia: {r.freq:.2f}<br>
    Plus Users: {r.plus_users_pct:.1%}<br>
    Core Users: {r.core_users_pct:.1%}<br>
    High Users: {r.high_users_pct:.1%}<br>
    Penetracion Dmart: {r.groceries_penetration:.1%}
    """
    folium.Marker(
        [r.lat_centro, r.lon_centro],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color='purple', icon='map-marker', prefix='fa'),
    ).add_to(fg_zonas)
fg_zonas.add_to(mapa)

# --- Capa 6: tiendas (poligono real o circulo fallback + popup con metricas) ---
fg_tiendas = folium.FeatureGroup(name='Tiendas', show=True)
_con_poligono, _con_circulo = 0, 0
for r in df_tiendas.itertuples():
    if pd.isna(r.store_lat) or pd.isna(r.store_lon):
        continue

    if pd.notna(r.penetracion):
        popup_html = f"""
        <b>{r.warehouse_name}</b><br>
        Penetracion: {r.penetracion:.1%}<br>
        MAUs cubiertos: {r.users_cubiertos:,.0f}<br>
        Ordenes: {r.ordenes:,.0f}<br>
        AFV: EUR {r.afv:,.2f}<br>
        Basket Size: {r.basket_size:.2f} u/orden
        """
    else:
        popup_html = f"<b>{r.warehouse_name}</b><br>Sin datos de cobertura este mes."

    popup = folium.Popup(popup_html, max_width=250)

    if pd.notna(r.shape_geojson):
        folium.GeoJson(
            json.loads(r.shape_geojson),
            style_function=lambda _: {'color': '#1a9850', 'weight': 2, 'fillOpacity': 0.10},
        ).add_child(folium.Popup(popup_html, max_width=250)).add_to(fg_tiendas)
        _con_poligono += 1
    else:
        folium.Circle(
            [r.store_lat, r.store_lon], radius=SERVICE_RADIUS_KM * 1000,
            color='#4575b4', weight=2, fill=True, fill_opacity=0.10,
        ).add_child(folium.Popup(popup_html, max_width=250)).add_to(fg_tiendas)
        _con_circulo += 1

    folium.Marker(
        [r.store_lat, r.store_lon],
        popup=popup,
        icon=folium.Icon(color='green', icon='shopping-cart', prefix='fa'),
    ).add_to(fg_tiendas)

fg_tiendas.add_to(mapa)
print(f'Tiendas con poligono real: {_con_poligono} | con circulo fallback: {_con_circulo}')

folium.LayerControl(collapsed=False).add_to(mapa)
mapa   # render inline; guardar con mapa.save("heatmap_pais.html")


[nota] Grilla recortada a las 20,000 celdas de mayor GMV PeYa (de 21,488 originales).
Tiendas con poligono real: 69 | con circulo fallback: 0
